> Experiments with [model.signal](../trapallada/model/signal.py)  module

In [1]:
import pathlib

import yaml

import jax.random as jr

import trapallada.data
import trapallada.model.nn
import trapallada.util
import trapallada.core

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


A *PyTorch* `Dataset` is built from a *sample* parameters

In [3]:
ds = trapallada.data.SignalsDataset(**trapallada.data.sample_dataset_parameters())
len(ds), ds[0][1].shape

(30856, torch.Size([5, 225]))

A parameters file that specifies an experiment

In [4]:
parameters_file = pathlib.Path.cwd().parent / 'parameters.yaml'
# parameters_file = pathlib.Path.cwd().parent / 'samples' / 'fft_vanilla_mini.yaml'
assert parameters_file.exists()

It is read into a `dict`

In [5]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [6]:
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
loss:
  x_sd: 0.15 # <---- FFT
  beta: 1.
model: &model
  channels:
  - 12
  - 12
  - 12
  dim_z: 40
  kernel_sizes: # <---- FFT (due to redundancy)
  - 64
  - 32
  - 16
  # stride: 2
  stride:
  - 2
  - 2
  - 2
data:
    fft: false # <---- FFT
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 2
  
  # ratio of the dataset to use for validation
  train_test_split:
    test_size: 0.2
  
  # whether or not to ignore non "good time intervals" (GTIs) in the loss function; it only
  # makes sense when working with time-domain signals, and will be ignored for frequency-domain ones
  mask_non_gti_intervals: true
VAE:
  class: VAE
  encoder:
    class: Encoder
    parameters:
      << : [*model]
  decoder:
    class: Decoder # <---- FFT (NonNegativeDecoder/NonNegativeConditionalDe

# Data

`DataLoaders` are derived from the `Dataset`

In [7]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


How many samples per batch?

In [8]:
batch = next(iter(train_dl))
len(batch)

3

The *index component* in the first batch

In [9]:
batch[0]

tensor([15587, 17950, 13864,  9754, 15479, 22831,  9973,  1757, 21462, 14604,
        18939,  5119, 29284,  8863, 28985,  5223, 23163, 28709,  4945, 15829,
        30348, 21295, 11209, 15240, 20197, 23916,  9995, 10546, 13858, 15190,
        14578, 28516])

The *x component* in the 1st element of the `Dataset`

In [10]:
signals_len = ds[0][1].shape
signals_len

torch.Size([5, 225])

What are the dimensions of every input?

In [11]:
tuple(ds[0][1].shape)

(5, 225)

Both *encoder* and *decoder* also require the *input size* as a parameter: it's inferred from the `Dataset` and included in the corresponding `dict`s

In [12]:
input_size = list(ds[0][1].shape)
for m in ['encoder', 'decoder']:
        parameters['VAE'][m]['parameters']['input_size'] = input_size

# Encoder

In [13]:
x = batch[1][0].numpy()
encoder = trapallada.model.nn.Encoder(
    **parameters['VAE']['encoder']['parameters'],
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

initial shape: (5, 225)
after equinox.nn._conv: (12, 81)
after jax._src.nn.functions: (12, 81)
after equinox.nn._normalisation: (12, 81)
after equinox.nn._conv: (12, 25)
after jax._src.nn.functions: (12, 25)
after equinox.nn._normalisation: (12, 25)
after equinox.nn._conv: (12, 5)
after jax._src.nn.functions: (12, 5)
after equinox.nn._normalisation: (12, 5)
after jax.numpy: (60,)
after equinox.nn._linear: (80,)
after trapallada.model.nn: (80,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,5,64],
      bias=f32[12,1],
      in_channels=5,
      out_channels=12,
      kernel_size=(64,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True,
      padding_mode='ZEROS'
    ),
    <PjitFunction of <function relu at 0x7f95d6849da0>>,
    LayerNorm(
      shape=(12, 81),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,81],
      bias=f32[12,81]
    ),
    Conv1d(
      num_spatial_d

In [14]:
x.shape

(5, 225)

In [15]:
z = encoder(x)
z.shape

(80,)

# Decoder

In [16]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.nn.Decoder(
    **(parameters['VAE']['decoder']['parameters'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

after equinox.nn._linear: (60,)
after jax._src.nn.functions: (60,)
after trapallada.model.nn: (12, 5)
after equinox.nn._conv: (12, 25)
after jax._src.nn.functions: (12, 25)
after equinox.nn._normalisation: (12, 25)
after equinox.nn._conv: (12, 81)
after jax._src.nn.functions: (12, 81)
after equinox.nn._normalisation: (12, 81)
after equinox.nn._conv: (5, 225)


In [17]:
vae = trapallada.model.nn.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((5, 225), (40,), (40,))

Training

In [18]:
model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)

trapallada.core - INFO - model has 34,117 parameters


  training                         validation                        
      loss loglikelihood        kl       loss loglikelihood        kl
0   8.2955     -0.499836  7.795664   1.268381     -0.482803  0.785578
   training                         validation                        
       loss loglikelihood        kl       loss loglikelihood        kl
1  0.799654     -0.488272  0.311383   0.628372     -0.481304  0.147068


# Conditional VAE

See [A deep dive into conditional variational autoencoders](https://beckham.nz/2023/04/27/conditional-vaes.html) (also [1](https://towardsdatascience.com/conditional-variational-autoencoders-with-learnable-conditional-embeddings-e22ee5359a2a) and [2](https://towardsdatascience.com/understanding-conditional-variational-autoencoders-cd62b4f57bf8))

In [19]:
cond_decoder = trapallada.model.nn.ConditionalDecoder(
    **(parameters['VAE']['decoder']['parameters'] | {'input_size': [1, input_size[1]]}| {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))

In [20]:
x_pred = cond_decoder(z_mean, x[-1:])
x_pred.shape

(1, 225)

Notice the **encoder is the same** as before.

In [21]:
cvae = trapallada.model.nn.CVAE(encoder, cond_decoder)
x_pred, z_mean, z_sd = cvae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 225), (40,), (40,))

In [22]:
import pandas as pd
pd.Timedelta(1.46, 'seconds')

Timedelta('0 days 00:00:01.460000')

# GMEncoder

In [23]:
encoder_parameters = parameters['VAE']['encoder']['parameters'].copy()
del encoder_parameters['dim_z']
encoder_parameters['dim_y'] = 3
encoder_parameters

{'channels': [12, 12, 12],
 'kernel_sizes': [64, 32, 16],
 'stride': [2, 2, 2],
 'input_size': [5, 225],
 'dim_y': 3}

In [24]:
# x = batch[1][0].numpy()
# encoder = trapallada.model.nn.GMEncoder(
#     **encoder_parameters,
#     key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
# # encoder.debug_call(x)
# # print(encoder)
# y = encoder(x)
# # assert y.sum() == 1
# y

In [25]:
# encoder.latent_given_class_and_data

In [26]:
# import equinox as eqx
# encoder.class_given_data
# # eqx.tree_at(encoder.class_given_data, 2)
# # eqx.filter(encoder, (True, [True]))
# eqx.tree_at(lambda m: m.layers, encoder, replace=encoder.layers[:-2])

In [27]:
# encoder.latent_given_class_and_data